# 浩劫原生自对弈训练
在 Colab 选择 **L4 / A100 或其他支持原生 BF16 的 GPU**。T4、P100 不支持本训练链路。

依次安装、查看自动配置，再将启动单元的 `START_TRAINING` 改为 `True`。默认不启动训练。Python 只负责安装和控制，采集、推理、学习与保存全部运行在原生程序中。

源码与依赖放在 `/content`；运行目录和定时检查点放在 Google Drive。断开或回收运行时后，从最近的完整 `checkpoint.pt` 恢复到新目录。保存间隔默认 300 秒，也可手动保存或停止。

最后一个单元显示 Web 控制台：吞吐、损失、策略滞后、资源曲线、保存/停止/并发控制，以及手工和定期 AI 测评。默认每 30 分钟测评一轮（12 局），可修改或关闭。单元会立即返回，浏览器每 5 秒刷新；关闭面板不停止训练或定期测评。

In [ ]:
from pathlib import Path
import json
import subprocess
from datetime import datetime, timezone

REPOSITORY = 'https://github.com/Loremmlel/haojie2.git'
REVISION = 'codex/bf16-elastic-training'  # 也可填已推送的提交号
SOURCE = Path('/content/haojie2')
if not SOURCE.exists():
    subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY, str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', 'origin', REVISION], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], check=True)
subprocess.run(['bash', str(SOURCE / 'training/colab/setup.sh')], check=True)
NATIVE = ['bash', str(SOURCE / 'training/native/run.sh')]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/haojie-training')
RESUME = ''  # 恢复时填旧运行目录中的 checkpoint.pt 完整路径
OVERRIDES = []  # 例如 ['--concurrency', '16']；通常使用自动配置
RUN = ROOT / datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
ARGS = ['--device', 'cuda', *OVERRIDES]
if RESUME:
    ARGS += ['--resume', RESUME]
configuration = json.loads(subprocess.check_output([*NATIVE, 'configure', *ARGS], text=True))
print(json.dumps(configuration, ensure_ascii=False, indent=2))
print('新运行目录:', RUN)

In [ ]:
START_TRAINING = False
if START_TRAINING:
    existing_process = globals().get('process')
    if existing_process is not None and existing_process.poll() is None:
        raise RuntimeError('已有训练进程运行，请先在控制单元停止。')
    if RUN.exists():
        raise RuntimeError('运行目录已存在，请重新执行配置单元生成新目录。')
    ROOT.mkdir(parents=True, exist_ok=True)
    LOG = RUN.with_suffix('.log')
    with LOG.open('x') as output:
        process = subprocess.Popen([*NATIVE, 'run', '--directory', str(RUN), *ARGS],
                                   stdout=output, stderr=subprocess.STDOUT)
    print('已启动，PID:', process.pid, '\n日志:', LOG)
else:
    print('尚未启动。确认配置后将 START_TRAINING 改为 True，再运行此单元。')

In [ ]:
import sys
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
from training.colab import panel

web_panel = panel.show(RUN, globals().get('process'), configuration,
                       native=NATIVE, python='/content/haojie-dependencies/venv/bin/python')